# 1. Preparing your data

`clashless` schedules presentations without double-booking anyone. Before it can help, it needs
three small tables describing your conference:

| Table | What it says |
|---|---|
| **presentations** | who's involved in each presentation, and who chairs (moderates) the session |
| **session times** | what time each session of the day starts |
| **unavailability** | who can't present at certain times |

This tutorial builds a tiny example of each and saves them as CSV files — the same three files
[tutorial 2](02_creating_a_schedule.ipynb) uses to actually build a schedule.

## Presentations

Each presentation needs four names, in fully symmetric roles: three **participants**
(`participant_1`/`participant_2`/`participant_3`) and a **chair** who runs that session. None of
the four is special — clashless doesn't distinguish "the presenter" from anyone else beyond the
column name, and the same person can appear in more than one role. Let's describe six
presentations for a small conference.

In [1]:
import pathlib

import pandas as pd

data_dir = pathlib.Path("data/small_conference")
data_dir.mkdir(parents=True, exist_ok=True)

columns = ["id", "participant_1", "participant_2", "participant_3", "chair"]
presentations = pd.DataFrame(
    [
        ["p1", "Alice Kim", "Daniel Ortiz", "Grace Liu", "Daniel Ortiz"],
        ["p2", "Ben Souza", "Grace Liu", "Henry Park", "Maria Novak"],
        ["p3", "Chloe Dubois", "Ian Ferreira", "Henry Park", "Maria Novak"],
        ["p4", "David Kaur", "Daniel Ortiz", "Ian Ferreira", "Maria Novak"],
        ["p5", "Elena Popescu", "Grace Liu", "Ian Ferreira", "Daniel Ortiz"],
        ["p6", "Farid Hossain", "Henry Park", "Grace Liu", "Daniel Ortiz"],
    ],
    columns=columns,
).set_index("id")

presentations

,participant_1,participant_2,participant_3,chair
id,,,,
p1,Alice Kim,Daniel Ortiz,Grace Liu,Daniel Ortiz
p2,Ben Souza,Grace Liu,Henry Park,Maria Novak
p3,Chloe Dubois,Ian Ferreira,Henry Park,Maria Novak
p4,David Kaur,Daniel Ortiz,Ian Ferreira,Maria Novak
p5,Elena Popescu,Grace Liu,Ian Ferreira,Daniel Ortiz
p6,Farid Hossain,Henry Park,Grace Liu,Daniel Ortiz


Nothing about repetition is validated here: the same person can appear in more than one role for
the same presentation — Daniel Ortiz chairs `p1`, which he's also a participant in, and that's
perfectly fine. Two presentations can also share any person in any combination.

`clashless` itself only ever works with `pandas` objects, never files directly — reading a CSV is
`pandas`'s job, not `clashless`'s. Let's save this table, then load it back with `pandas` and wrap
it with `clashless.Presentations` (it assumes the index you give it is already correct - here,
`pd.read_csv`'s `index_col=0` puts `id` back where it belongs).

In [2]:
import clashless as cl

presentations.to_csv(data_dir / "presentations.csv")

loaded = cl.Presentations(pd.read_csv(data_dir / "presentations.csv", index_col=0))
loaded.data

,participant_1,participant_2,participant_3,chair
id,,,,
p1,Alice Kim,Daniel Ortiz,Grace Liu,Daniel Ortiz
p2,Ben Souza,Grace Liu,Henry Park,Maria Novak
p3,Chloe Dubois,Ian Ferreira,Henry Park,Maria Novak
p4,David Kaur,Daniel Ortiz,Ian Ferreira,Maria Novak
p5,Elena Popescu,Grace Liu,Ian Ferreira,Daniel Ortiz
p6,Farid Hossain,Henry Park,Grace Liu,Daniel Ortiz


### Checking your data

Since nothing here is validated, it's easy to accidentally reuse a name in ways you didn't
intend — a typo that quietly turns two different people into one, say. `clashless.report` prints
a quick summary instead: how unique each role column is, how often the same person holds two
roles within one presentation, and who shows up the most overall.

In [3]:
cl.isvalid.report(loaded)

Presentation role repetition report
Rows: 6

Per-column uniqueness:
  participant_1: 6 unique / 6 rows (0 repeated)
  participant_2: 4 unique / 6 rows (2 repeated)
  participant_3: 3 unique / 6 rows (3 repeated)
  chair: 2 unique / 6 rows (4 repeated)

Repeats within the same presentation (same person, same row):
  participant_1 == participant_2: 0 row(s)
  participant_1 == participant_3: 0 row(s)
  participant_1 == chair: 0 row(s)
  participant_2 == participant_3: 0 row(s)
  participant_2 == chair: 1 row(s)
  participant_3 == chair: 0 row(s)
  rows with any within-row repeat: 1

Most frequently appearing people (top 10):
  Daniel Ortiz: 5
  Grace Liu: 4
  Ian Ferreira: 3
  Henry Park: 3
  Maria Novak: 3
  Alice Kim: 1
  Ben Souza: 1
  Chloe Dubois: 1
  David Kaur: 1
  Elena Popescu: 1


## Session times

`clashless` doesn't need a list of calendar dates — just how many sessions happen each day and
when they start. The *same* sessions repeat on every conference day, so there's no `day` column
here at all; "day" only appears later, in the schedule `clashless` produces.

Our small conference has 3 sessions a day.

In [4]:
session_times = pd.DataFrame(
    {"session": [1, 2, 3], "start_time": ["09:00", "11:00", "14:00"]}
).set_index("session")
session_times.to_csv(data_dir / "session-start-times.csv")

loaded_session_times = cl.SessionTimes(
    pd.read_csv(data_dir / "session-start-times.csv", index_col="session")
)
print(f"{len(loaded_session_times)} sessions per day")
loaded_session_times.data

3 sessions per day


,start_time
session,
1,09:00
2,11:00
3,14:00


## Unavailability

This table lists exceptions: times when a specific person can't present. Each row has a `person`,
and a `day` and `session` that can each be left blank — a blank acts as a wildcard meaning "any
value here." That gives four kinds of rule from the same two columns:

| `day` | `session` | Meaning |
|---|---|---|
| set | blank | unavailable **all day**, that one day |
| blank | set | unavailable during that **session, every day** |
| set | set | unavailable for that **one specific slot** only |
| blank | blank | unavailable for the **entire conference** |

Our example uses the first three:

In [5]:
unavailable = pd.DataFrame(
    [
        ["Daniel Ortiz", 1, None],  # unavailable all day on day 1
        ["Grace Liu", None, 2],  # unavailable during session 2, every day
        ["Henry Park", 2, 3],  # unavailable specifically on day 2, session 3
    ],
    columns=["person", "day", "session"],
)
unavailable.to_csv(data_dir / "unavailable.csv", index=False)

loaded_unavailable = cl.Unavailability(
    pd.read_csv(
        data_dir / "unavailable.csv", dtype={"day": "Int64", "session": "Int64"}
    )
)
loaded_unavailable.data

,person,day,session
0,Daniel Ortiz,1,<NA>
1,Grace Liu,<NA>,2
2,Henry Park,2,3


`Unavailability.is_unavailable(person, day, session)` answers "is this person free at this
slot?" — it's what the solver checks internally. Here it is confirming the rules above, plus a
quick look at the fourth kind (`day` and `session` both blank), which we didn't add to our main
example since one person being unavailable for the *entire* conference only makes sense if they
aren't essential to any presentation.

In [6]:
def check(person, day, session):
    """Print whether `person` is free or unavailable at (day, session)."""
    unavailable = loaded_unavailable.is_unavailable(person, day, session)
    status = "unavailable" if unavailable else "free"
    print(f"{person}, day {day}, session {session}: {status}")


check("Daniel Ortiz", 1, 1)
check("Daniel Ortiz", 2, 1)
check("Grace Liu", 3, 2)
check("Henry Park", 2, 3)
check("Henry Park", 2, 1)

everywhere_unavailable = cl.Unavailability(
    pd.DataFrame({"person": ["Guest Speaker"], "day": [None], "session": [None]})
)
is_unavailable = everywhere_unavailable.is_unavailable("Guest Speaker", 1, 1)
print(is_unavailable, "(entire conference)")

Daniel Ortiz, day 1, session 1: unavailable
Daniel Ortiz, day 2, session 1: free
Grace Liu, day 3, session 2: unavailable
Henry Park, day 2, session 3: unavailable
Henry Park, day 2, session 1: free
True (entire conference)


## Wrap-up

- `Presentations`, `SessionTimes`, and `Unavailability` each wrap a `pandas.DataFrame` (`SessionTimes`
  also accepts a `pandas.Series`) that's already indexed correctly — loading from a file is
  `pandas`'s job (`pd.read_csv(...)`), not `clashless`'s.
- A presentation's four roles (`participant_1`/`participant_2`/`participant_3`/`chair`) are fully
  symmetric — nothing about repetition is validated. `clashless.isvalid.report` gives you a quick
  summary of whatever repetition your own data actually has.
- `unavailable.csv`'s blank `day`/`session` combinations give you four rule types from two columns.

These three files are now saved under `data/small_conference/`. Next:
[2. Creating a schedule](02_creating_a_schedule.ipynb) loads them and produces an actual timetable.